# Synthetic Data Generation for Tool-Calling Fine-Tuning

Fine-tune the exact Azure AI Foundry model `OpenAI-OSS / gpt-oss-120b / version 1` to call the bundled catalog correctly. This run is intentionally **training-only**: it generates data, preserves the train/validation/test contract, submits a `GlobalStandard` job, monitors it to a terminal state, downloads result files, and reports validation loss/token accuracy. It does not create or call a base or fine-tuned deployment.

End-to-end:
1. Convert the unchanged OpenAI-format 15-tool catalog to OpenAPI 3.0 and upload it
2. Request three synthetic-data batches with `max_samples=60` per batch
3. Deduplicate and split the resulting examples 80% / 10% / 10%
4. Submit `gpt-oss-120b` supervised fine-tuning for 3 epochs at learning-rate multiplier 1.0
5. Monitor the job to a terminal state and download service result files
6. Summarize training and validation loss/token accuracy only

The README previously described "~50" generated examples and called the fixture a 6-tool catalog. The notebook's actual default contract is three 60-sample requests and the fixture contains 15 tools. The service may return fewer samples than requested; the exact returned, deduplicated rows are the rows split and uploaded.

## 1. Setup & Configuration

In [ ]:
import csv, os, json, time, random
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

PROJECT_ENDPOINT = os.environ.get("AZURE_AI_PROJECT_ENDPOINT")
if not PROJECT_ENDPOINT:
    raise ValueError("Set AZURE_AI_PROJECT_ENDPOINT to your Microsoft Foundry training project endpoint")
GENERATION_PROJECT_ENDPOINT = os.environ.get(
    "AZURE_GENERATION_PROJECT_ENDPOINT",
    PROJECT_ENDPOINT,
)
TEACHER_MODEL = os.environ.get("AZURE_TEACHER_DEPLOYMENT", "gpt-4.1")
STUDENT_MODEL = os.environ.get("AZURE_STUDENT_MODEL", "gpt-oss-120b")
STUDENT_MODEL_FORMAT = "OpenAI-OSS"
STUDENT_MODEL_VERSION = "1"
TRAINING_TYPE = os.environ.get("AZURE_FINE_TUNING_TRAINING_TYPE", "GlobalStandard")
RESUME_FT_JOB_ID = os.environ.get("AZURE_FINE_TUNING_JOB_ID")
RUN_ID = os.environ.get("LOOM_RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))

if TRAINING_TYPE != "GlobalStandard":
    raise ValueError(f"This run requires GlobalStandard, not {TRAINING_TYPE!r}")
if "eastus2" in PROJECT_ENDPOINT.lower() or "eastus2" in GENERATION_PROJECT_ENDPOINT.lower():
    raise ValueError("East US 2 is prohibited for this run")
if STUDENT_MODEL != "gpt-oss-120b":
    raise ValueError(f"Exact student model gpt-oss-120b is required, not {STUDENT_MODEL!r}")

OUTPUT_ROOT = Path("./outputs/loom-model-runs").resolve()
WORK = OUTPUT_ROOT / RUN_ID
WORK.mkdir(parents=True, exist_ok=True)

credential = DefaultAzureCredential()
project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
    polling_interval=15,
)
model_client = project.get_openai_client()
generation_project = AIProjectClient(
    endpoint=GENERATION_PROJECT_ENDPOINT,
    credential=credential,
    polling_interval=15,
)
generation_client = generation_project.get_openai_client()

RUN_STATE = {
    "run_id": RUN_ID,
    "project_endpoint": PROJECT_ENDPOINT,
    "generation_project_endpoint": GENERATION_PROJECT_ENDPOINT,
    "auth": "DefaultAzureCredential",
    "teacher_model": TEACHER_MODEL,
    "student_model": {
        "format": STUDENT_MODEL_FORMAT,
        "name": STUDENT_MODEL,
        "version": STUDENT_MODEL_VERSION,
    },
    "training_type": TRAINING_TYPE,
    "deployment_and_inference": "intentionally_not_performed",
    "default_contract": {
        "generation_batches": 3,
        "max_samples_per_batch": 60,
        "split": {"train": 0.8, "validation": 0.1, "test": 0.1},
        "n_epochs": 3,
        "learning_rate_multiplier": 1.0,
        "tool_count": 15,
    },
    "readme_notebook_discrepancies": [
        "README said approximately 50 generated examples; notebook requests 3 x 60 samples.",
        "README file table said 6 tools; the fixture and notebook contain 15 tools.",
        "Service output can be lower than max_samples; exact returned unique rows are used.",
    ],
    "sdk_usage": {
        "project": "AIProjectClient",
        "model_and_fine_tuning": "AIProjectClient.get_openai_client",
        "data_generation": "AIProjectClient.beta.datasets poller",
    },
    "stages": {},
}

print(f"Training project:   {PROJECT_ENDPOINT}")
print(f"Generation project: {GENERATION_PROJECT_ENDPOINT}")
print(f"Teacher deployment: {TEACHER_MODEL}")
print(f"Student model:      {STUDENT_MODEL_FORMAT}/{STUDENT_MODEL}/{STUDENT_MODEL_VERSION}")
print(f"Training type:      {TRAINING_TYPE}")
print(f"Work dir:           {WORK}")

## 2. Inline helpers: tool catalog conversion + Foundry data generation

The conversion helper maps the OpenAI-format tool array to the OpenAPI 3.0 input required by `ToolUseFineTuning`. Data generation uses the SDK's documented `DatasetGenerationLROPoller`; the data-generation operation is still under `project.beta.datasets` in `azure-ai-projects` 2.x.

In [ ]:
def openai_tools_to_openapi(openai_tools, title="Tools"):
    """Convert an OpenAI chat-completions tools array to an OpenAPI 3.0 spec."""
    def clean_schema(value):
        if isinstance(value, dict):
            cleaned = {key: clean_schema(item) for key, item in value.items()}
            if cleaned.get("required") == []:
                cleaned.pop("required")
            return cleaned
        if isinstance(value, list):
            return [clean_schema(item) for item in value]
        return value

    paths = {}
    for tool in openai_tools:
        function = tool.get("function", {})
        name = function["name"]
        parameters = clean_schema(
            function.get("parameters") or {"type": "object", "properties": {}}
        )
        paths[f"/{name}"] = {
            "post": {
                "operationId": name,
                "summary": function.get("description", ""),
                "requestBody": {
                    "required": True,
                    "content": {"application/json": {"schema": parameters}},
                },
                "responses": {"200": {"description": "Success"}},
            }
        }
    return {
        "openapi": "3.0.3",
        "info": {"title": title, "version": "1.0.0"},
        "paths": paths,
    }


def submit_foundry_datagen(*, file_id, teacher, max_samples, output_name, timeout_s=1800):
    """Submit ToolUseFineTuning generation and download its exact JSONL output."""
    from azure.ai.projects.models import (
        DataGenerationJob,
        DataGenerationJobInputs,
        DataGenerationJobScenario,
        DataGenerationJobOutputOptions,
        ToolUseFineTuningDataGenerationJobOptions,
        FileDataGenerationJobSource,
        DataGenerationModelOptions,
    )

    options = ToolUseFineTuningDataGenerationJobOptions(
        max_samples=max_samples,
        model_options=DataGenerationModelOptions(model=teacher),
    )
    job = DataGenerationJob(inputs=DataGenerationJobInputs(
        name=output_name,
        scenario=DataGenerationJobScenario.SUPERVISED_FINETUNING,
        sources=[FileDataGenerationJobSource(id=file_id)],
        options=options,
        output_options=DataGenerationJobOutputOptions(name=output_name),
    ))

    print(f"Submitting data-generation job (max_samples={max_samples})...")
    poller = generation_project.beta.datasets.begin_create_generation_job(job=job)
    job_id = poller.details.get("job_id")
    if not job_id:
        raise RuntimeError("Data-generation submission did not return a job ID")
    RUN_STATE.setdefault("data_generation_job_ids", []).append(job_id)
    print(f"  job.id={job_id}")

    result = poller.result(timeout=timeout_s)
    outputs = result.outputs or []
    if not outputs:
        raise RuntimeError(f"Data-generation job {job_id} completed without output files")

    output_path = WORK / f"{output_name}_dg.jsonl"
    with output_path.open("wb") as destination:
        for output in outputs:
            destination.write(generation_client.files.content(file_id=output.id).read())
    RUN_STATE.setdefault("data_generation_statuses", []).append(str(poller.status()))
    RUN_STATE.setdefault("data_generation_returned_samples", []).append(result.generated_samples)
    print(f"  generated {result.generated_samples} sample(s) in {len(outputs)} file(s)")
    print(f"  saved {output_path.name} ({output_path.stat().st_size:,} bytes)")
    return output_path

## 3. Convert + upload the Zava tool catalog

The bundled fixture contains 15 retail-support tools. The notebook converts it to OpenAPI 3.0 and uploads it through the model/files client obtained from the project client.

In [ ]:
OPENAI_TOOLS = json.loads(Path("fixtures/zava_tools_openai.json").read_text(encoding="utf-8"))
if len(OPENAI_TOOLS) != 15:
    raise RuntimeError(f"Expected the unchanged 15-tool catalog, found {len(OPENAI_TOOLS)}")
print(f"Loaded {len(OPENAI_TOOLS)} tools: {[tool['function']['name'] for tool in OPENAI_TOOLS]}")

openapi_spec = openai_tools_to_openapi(OPENAI_TOOLS, title="Zava Resolution Desk Tools")
OPENAPI_OUT = WORK / "zava_tools_openapi.json"
OPENAPI_OUT.write_text(json.dumps(openapi_spec, indent=2), encoding="utf-8")

with OPENAPI_OUT.open("rb") as file_handle:
    spec_file = generation_client.files.create(
        file=(OPENAPI_OUT.name, file_handle),
        purpose="user_data",
    )

for _ in range(20):
    spec_file = generation_client.files.retrieve(file_id=spec_file.id)
    if spec_file.status in ("processed", "error"):
        break
    time.sleep(2)
if spec_file.status != "processed":
    raise RuntimeError(f"Tool spec {spec_file.id} ended in status {spec_file.status}")
RUN_STATE["tool_catalog_file_id"] = spec_file.id
RUN_STATE["tool_names"] = [tool["function"]["name"] for tool in OPENAI_TOOLS]
print(f"Uploaded: {spec_file.id} (status={spec_file.status})")

## 4. Generate synthetic training data

The actual notebook default is three independent `ToolUseFineTuning` requests with `max_samples=60` each. This preserves the original generation scale exactly. The service may return fewer than 60 rows per request; all returned rows are downloaded, deduplicated, and recorded rather than padded or substituted.

In [ ]:
DATAFILES = [WORK / f"zava-tools-sft-{batch}_dg.jsonl" for batch in range(1, 4)]
reuse_generated = os.environ.get("AZURE_REUSE_GENERATED_DATA", "").lower() in {"1", "true", "yes"}
if reuse_generated:
    missing = [str(path) for path in DATAFILES if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Requested exact generated-data reuse, but files are missing: {missing}")
    RUN_STATE["data_generation_reuse"] = "exact outputs recovered after a downstream upload failure"
    print("Reusing the exact three generated outputs from this run after downstream upload failure.")
else:
    DATAFILES = [
        submit_foundry_datagen(
            file_id=spec_file.id,
            teacher=TEACHER_MODEL,
            max_samples=60,
            output_name=f"zava-tools-sft-{batch}",
        )
        for batch in range(1, 4)
    ]

data = []
seen = set()
for datafile in DATAFILES:
    with datafile.open(encoding="utf-8") as source_file:
        for line in source_file:
            if not line.strip():
                continue
            row = json.loads(line)
            key = json.dumps(row, sort_keys=True)
            if key not in seen:
                seen.add(key)
                data.append(row)
if len(data) < 20:
    raise RuntimeError(f"Data generation produced only {len(data)} unique rows; at least 20 are required")
RUN_STATE["stages"]["data_generation"] = "succeeded"
RUN_STATE["generated_rows"] = len(data)
print()
print(f"Unique rows: {len(data)}")
print("First row preview:")
print(json.dumps(data[0]["messages"][:2], indent=2)[:500])

## 5. Split into train / validation / test

Preserve the notebook's deterministic seed-42 split: 80% training, 10% validation, and 10% held-out test. The test rows are saved as part of the original contract but are not used for deployment or inference evaluation in this training-only run.

In [ ]:
rng = random.Random(42)
indices = list(range(len(data)))
rng.shuffle(indices)
n_train = int(0.8 * len(indices))
n_val = int(0.1 * len(indices))
train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

TRAIN_PATH = WORK / "train.jsonl"
VAL_PATH = WORK / "val.jsonl"
TEST_PATH = WORK / "test.jsonl"
for path, selected_indices in (
    (TRAIN_PATH, train_idx),
    (VAL_PATH, val_idx),
    (TEST_PATH, test_idx),
):
    with path.open("w", encoding="utf-8") as destination:
        for index in selected_indices:
            destination.write(json.dumps(data[index]) + "\n")

RUN_STATE["dataset_scale"] = {
    "generated_unique": len(data),
    "train": len(train_idx),
    "validation": len(val_idx),
    "test": len(test_idx),
}
print(f"  train: {len(train_idx)} rows")
print(f"  val:   {len(val_idx)} rows")
print(f"  test:  {len(test_idx)} rows")

## 6. Validate the generated training contract

Deployment-based baseline evaluation is intentionally removed. This cell verifies only the exact generated split and tool-call-bearing training format before upload.

In [ ]:
def count_tool_call_turns(path):
    rows = 0
    tool_call_turns = 0
    with path.open(encoding="utf-8") as source:
        for line in source:
            if not line.strip():
                continue
            rows += 1
            messages = json.loads(line).get("messages", [])
            tool_call_turns += sum(
                1
                for message in messages
                if message.get("role") == "assistant" and message.get("tool_calls")
            )
    return {"rows": rows, "assistant_tool_call_turns": tool_call_turns}


split_validation = {
    "train": count_tool_call_turns(TRAIN_PATH),
    "validation": count_tool_call_turns(VAL_PATH),
    "test": count_tool_call_turns(TEST_PATH),
}
if any(details["assistant_tool_call_turns"] == 0 for details in split_validation.values()):
    raise RuntimeError(f"Every split must contain tool-call supervision: {split_validation}")
RUN_STATE["split_validation"] = split_validation
RUN_STATE["stages"]["dataset_validation"] = "succeeded"
print(json.dumps(split_validation, indent=2))

## 7. Submit the `gpt-oss-120b` fine-tuning job

Preserve the notebook's actual hyperparameter defaults: **3 epochs** and **learning-rate multiplier 1.0**. The exact catalog model is `OpenAI-OSS / gpt-oss-120b / version 1`, submitted as `gpt-oss-120b` with `GlobalStandard`.

In [ ]:
if RESUME_FT_JOB_ID:
    print(f"Reusing fine-tuning job {RESUME_FT_JOB_ID}...")
    ft_job = model_client.fine_tuning.jobs.retrieve(RESUME_FT_JOB_ID)
    train_file = model_client.files.retrieve(file_id=ft_job.training_file)
    val_file = model_client.files.retrieve(file_id=ft_job.validation_file)
else:
    print("Uploading train + validation files...")
    with TRAIN_PATH.open("rb") as file_handle:
        train_file = model_client.files.create(
            file=(TRAIN_PATH.name, file_handle),
            purpose="fine-tune",
        )
    with VAL_PATH.open("rb") as file_handle:
        val_file = model_client.files.create(
            file=(VAL_PATH.name, file_handle),
            purpose="fine-tune",
        )

    uploaded_files = []
    for uploaded in (train_file, val_file):
        last_error = None
        for attempt in range(120):
            try:
                uploaded = model_client.files.retrieve(file_id=uploaded.id)
                last_error = None
            except Exception as error:
                last_error = error
                if getattr(error, "status_code", None) != 404:
                    raise
            if last_error is None and uploaded.status in ("processed", "error"):
                break
            time.sleep(min(10, 2 + attempt // 10))
        if last_error is not None:
            raise RuntimeError(
                f"File {uploaded.id} did not propagate to the regional file API"
            ) from last_error
        if uploaded.status != "processed":
            raise RuntimeError(f"File {uploaded.id} ended in status {uploaded.status}")
        uploaded_files.append(uploaded)
        print(f"  {uploaded.id} status={uploaded.status}")
    train_file, val_file = uploaded_files

    print(f"\nSubmitting {TRAINING_TYPE} fine-tuning job...")
    ft_job = model_client.fine_tuning.jobs.create(
        model=STUDENT_MODEL,
        training_file=train_file.id,
        validation_file=val_file.id,
        method={"type": "supervised"},
        hyperparameters={"n_epochs": 3, "learning_rate_multiplier": 1.0},
        suffix="zava-tools-gpt-oss-120b",
        extra_body={"trainingType": TRAINING_TYPE},
    )
RUN_STATE["training_file_id"] = train_file.id
RUN_STATE["validation_file_id"] = val_file.id
RUN_STATE["fine_tuning_job_id"] = ft_job.id
RUN_STATE["stages"]["fine_tuning_submission"] = "succeeded"
print(f"  Job: {ft_job.id}  status={ft_job.status}")

## 8. Monitor training to a terminal state

Poll every 30 seconds. Terminal failure is still summarized because service result files and events can contain completed training/validation evidence even when a later registration stage fails.

In [ ]:
print(f"Monitoring job {ft_job.id}...\n")
last_event_ids = set()
terminal_statuses = {"succeeded", "failed", "cancelled"}
while True:
    job = model_client.fine_tuning.jobs.retrieve(ft_job.id)
    events = list(model_client.fine_tuning.jobs.list_events(
        fine_tuning_job_id=ft_job.id,
        limit=100,
    ))
    for event in reversed(events):
        event_id = getattr(event, "id", None) or (
            getattr(event, "created_at", None),
            getattr(event, "message", None),
        )
        if event_id in last_event_ids:
            continue
        last_event_ids.add(event_id)
        message = event.message or ""
        if "Step " in message or "validation" in message.lower() or "training" in message.lower():
            print(f"  {time.strftime('%H:%M:%S')}  {message[:180]}")
    if job.status in terminal_statuses:
        print(f"\nFinal status: {job.status}")
        break
    time.sleep(30)

RUN_STATE["fine_tuning_status"] = job.status
RUN_STATE["fine_tuned_model_id"] = getattr(job, "fine_tuned_model", None)
RUN_STATE["fine_tuning_error"] = (
    getattr(getattr(job, "error", None), "message", None)
    if getattr(job, "error", None)
    else None
)
RUN_STATE["stages"]["fine_tuning"] = job.status

events_path = WORK / "fine_tuning_events.json"
events_path.write_text(
    json.dumps(
        [
            event.model_dump(mode="json")
            if hasattr(event, "model_dump")
            else {"message": getattr(event, "message", None)}
            for event in events
        ],
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)
print(f"Saved events: {events_path}")

## 9. Download result files and extract training metrics

No model deployment is created or validated. Download all service result files and extract the final/best reported training and validation loss/token-accuracy fields.

In [ ]:
result_file_ids = list(getattr(job, "result_files", None) or [])
downloaded_results = []
metric_rows = []
for result_file_id in result_file_ids:
    metadata = model_client.files.retrieve(file_id=result_file_id)
    filename = getattr(metadata, "filename", None) or f"{result_file_id}.bin"
    destination = WORK / filename
    destination.write_bytes(model_client.files.content(file_id=result_file_id).read())
    downloaded_results.append({
        "file_id": result_file_id,
        "filename": destination.name,
        "bytes": destination.stat().st_size,
    })
    if destination.suffix.lower() == ".csv":
        with destination.open(encoding="utf-8-sig", newline="") as source:
            metric_rows.extend(list(csv.DictReader(source)))

RUN_STATE["result_files"] = downloaded_results
print(json.dumps(downloaded_results, indent=2))

## 10. Training-only validation summary

Report service-produced loss and token-accuracy metrics only. The held-out test split remains saved but unused because base/fine-tuned deployment and inference evaluation are prohibited for this run.

In [ ]:
def numeric(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


metric_columns = sorted({
    key
    for row in metric_rows
    for key in row
    if any(token in key.lower() for token in ("loss", "accuracy", "step", "epoch"))
})
metrics_by_column = {
    column: [numeric(row.get(column)) for row in metric_rows]
    for column in metric_columns
}
metrics_by_column = {
    column: [value for value in values if value is not None]
    for column, values in metrics_by_column.items()
}
metrics_by_column = {
    column: {
        "first": values[0],
        "last": values[-1],
        "best_min": min(values) if "loss" in column.lower() else None,
        "best_max": max(values) if "accuracy" in column.lower() else None,
        "count": len(values),
    }
    for column, values in metrics_by_column.items()
    if values
}

RUN_STATE["training_metrics"] = metrics_by_column
RUN_STATE["validated_at_utc"] = datetime.now(timezone.utc).isoformat()
RUN_STATE["package_versions"] = {
    name: version(name)
    for name in ("azure-ai-projects", "azure-identity", "openai")
}
RUN_STATE["stages"]["result_download"] = "succeeded"
SUMMARY_PATH = WORK / "run_summary.json"
SUMMARY_PATH.write_text(json.dumps(RUN_STATE, indent=2), encoding="utf-8")

print("-" * 72)
print(f"Job:    {ft_job.id}")
print(f"Status: {job.status}")
print(f"Model:  {STUDENT_MODEL_FORMAT}/{STUDENT_MODEL}/{STUDENT_MODEL_VERSION}")
print(f"Rows:   {len(train_idx)} train / {len(val_idx)} validation / {len(test_idx)} test")
print("Metrics:")
print(json.dumps(metrics_by_column, indent=2))
print(f"Summary: {SUMMARY_PATH}")

if job.status != "succeeded":
    print(f"Terminal service error: {RUN_STATE['fine_tuning_error']}")

## Outputs

All ignored runtime artifacts are stored under `outputs/loom-model-runs/<run-id>/`:

- exact generated batch files
- deterministic train / validation / held-out test JSONL
- fine-tuning events
- downloaded service result files
- `run_summary.json`

This training-only notebook intentionally performs no base deployment, fine-tuned deployment, or inference evaluation.

## Dependencies

```bash
pip install "azure-ai-projects>=2.7,<3" "azure-identity>=1.26,<2" jupyter
```